In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from scipy.fft import fft, fftfreq

# ---------------------------------------------------------
# 1. PARÂMETROS FÍSICOS DO SISTEMA (ESP32-S3 REAL)
# ---------------------------------------------------------
TAXA_AMOSTRAGEM = 3200
TAMANHO_JANELA = 1024

# Fatores de conversão (Do Dado Bruto para a Física Real)
FATOR_ADXL_G = 0.0039          # Transforma o LSB do acelerômetro em Força G
FATOR_MIC_NORM = 8388608.0     # Transforma os 24-bits do INMP441 numa escala de -1.0 a 1.0

GRAVIDADE = 9.80665
REF_MIC = 1.0

dataset_ia = []
dataset_humano = []

print("Fundações Físicas configuradas para processar os dados BRUTOS do ESP32.")

# ---------------------------------------------------------
# 2.
# ---------------------------------------------------------
def mastigar_dados_god_payload(caminho_arquivo, label_defeito):

    df_bruto = pd.read_csv(caminho_arquivo)

    def extrair_vib_cplusplus(sinal_g):
        rms = np.sqrt(np.mean(sinal_g**2))
        kurt = kurtosis(sinal_g, fisher=False)
        skew_val = skew(sinal_g, bias=False)

        yf = np.abs(fft(sinal_g))
        xf = fftfreq(TAMANHO_JANELA, 1 / TAXA_AMOSTRAGEM)

        mag = yf[:TAMANHO_JANELA//2] / TAMANHO_JANELA
        energia = mag * mag
        freqs = xf[:TAMANHO_JANELA//2]

        e_b1, p_b1, e_b2, e_b3, e_b4 = 0.0, 0.0, 0.0, 0.0, 0.0

        for i in range(1, len(freqs)):
            f = freqs[i]
            eng = energia[i]
            m = mag[i]

            if f <= 60.0:
                e_b1 += eng
                if m > p_b1: p_b1 = m
            elif f <= 150.0:
                e_b2 += eng
            elif f <= 500.0:
                e_b3 += eng
            else:
                e_b4 += eng

        return [rms, kurt, skew_val, e_b1, p_b1, e_b2, e_b3, e_b4]

    # Janela deslizante de 1024 em 1024 linhas
    for inicio in range(0, len(df_bruto) - TAMANHO_JANELA, TAMANHO_JANELA):
        janela = df_bruto.iloc[inicio : inicio + TAMANHO_JANELA]

        # =========================================================
        #  Conversão do Bruto para o Físico
        # =========================================================
        sinal_z = janela['az'].values * FATOR_ADXL_G
        sinal_y = janela['ay'].values * FATOR_ADXL_G
        sinal_x = janela['ax'].values * FATOR_ADXL_G
        sinal_mic = janela['mic'].values / FATOR_MIC_NORM

        # 1. Extração de Vibração (3 eixos * 8 features = 24 features)
        feat_z = extrair_vib_cplusplus(sinal_z)
        feat_y = extrair_vib_cplusplus(sinal_y)
        feat_x = extrair_vib_cplusplus(sinal_x)

        # 2. Extração do Microfone
        mic_rms = np.sqrt(np.mean(sinal_mic**2))
        mic_pico = np.max(np.abs(sinal_mic))
        mic_crista = mic_pico / mic_rms if mic_rms > 0 else 0

        TAXA_MIC = 16000
        yf_mic = np.abs(fft(sinal_mic))
        xf_mic = fftfreq(TAMANHO_JANELA, 1 / TAXA_MIC)
        mag_mic = yf_mic[:TAMANHO_JANELA//2] / TAMANHO_JANELA
        eng_mic = mag_mic * mag_mic
        freqs_mic = xf_mic[:TAMANHO_JANELA//2]

        e = [0.0]*5
        p = [0.0]*5

        for i in range(1, len(freqs_mic)):
            f = freqs_mic[i]
            eng = eng_mic[i]
            m = mag_mic[i]

            b = -1
            if f <= 500: b = 0
            elif f <= 2000: b = 1
            elif f <= 5000: b = 2
            elif f <= 10000: b = 3
            else: b = 4

            if b != -1:
                e[b] += eng
                if m > p[b]: p[b] = m

        feat_mic = [mic_rms, mic_crista]
        for i in range(5):
            feat_mic.append(e[i])
            feat_mic.append(p[i])

        features_totais = feat_z + feat_y + feat_x + feat_mic

        # =========================================================
        # PARTE C: SALVANDO NOS BANCOS
        # =========================================================
        linha_ia = {f'feat_{i}': features_totais[i] for i in range(36)}
        linha_ia['label_original'] = label_defeito
        linha_ia['arquivo_origem'] = caminho_arquivo
        dataset_ia.append(linha_ia)

        linha_humana = {
            'Diagnostico_Real': label_defeito,
            'Acustica_dB': np.round(20 * np.log10((mic_rms + 1e-12) / REF_MIC), 1),
            'Eixo_Z_ms2': np.round(feat_z[0] * GRAVIDADE, 2),
            'Eixo_Y_ms2': np.round(feat_y[0] * GRAVIDADE, 2),
            'Eixo_X_ms2': np.round(feat_x[0] * GRAVIDADE, 2),
            'Arquivo': caminho_arquivo
        }
        dataset_humano.append(linha_humana)

print("Espelho C++ criado! O Python vai gerar 36 features alinhadas com a Inferência.")

# ---------------------------------------------------------
# 3. EXECUÇÃO E SALVAMENTO DOS BANCOS DE DADOS
# ---------------------------------------------------------
MAPA_DEFEITOS = {
    "normal": 0,
    "desbalanceamento": 1,
}

print("Iniciando varredura dupla (Extrator IA + Extrator Humano)...")

caminho_base = 'meu_dataset'

for pasta_chave, label_id in MAPA_DEFEITOS.items():
    padrao_busca = os.path.join(caminho_base, pasta_chave, '*.csv')
    arquivos_encontrados = glob.glob(padrao_busca)

    if len(arquivos_encontrados) > 0:
        print(f"-> Analisando {len(arquivos_encontrados)} arquivos da classe: {pasta_chave}")
        for caminho in arquivos_encontrados:
            mastigar_dados_god_payload(caminho, label_id)

# ---------------------------------------------------------
# FINALIZAÇÃO: SALVANDO NO DISCO
# ---------------------------------------------------------
if len(dataset_ia) > 0:
    df_ia = pd.DataFrame(dataset_ia)
    df_ia.to_csv('dataset_ia_hardware_real.csv', index=False)

    df_humano = pd.DataFrame(dataset_humano)
    df_humano.to_csv('dataset_dashboard_hardware_real.csv', index=False)

    print("\ TRABALHO CONCLUÍDO ")
    print(f"Dataset IA (Hardware Real): {len(df_ia)} linhas prontas para treinar o XGBoost.")
else:
    print("\nNenhum arquivo encontrado. Verifique se a pasta 'meu_dataset' está no mesmo local deste script.")

✅ Fundações Físicas configuradas para processar os dados BRUTOS do ESP32.
✅ Espelho C++ criado! O Python vai gerar EXATAS 36 features alinhadas com a Inferência.
Iniciando varredura dupla (Extrator IA + Extrator Humano)...
-> Analisando 2 arquivos da classe: normal
-> Analisando 2 arquivos da classe: desbalanceamento

🎉 TRABALHO CONCLUÍDO COM SUCESSO ABSOLUTO!
🧠 Dataset IA (Hardware Real): 3735 linhas prontas para treinar o XGBoost.


In [ ]:
!pip install xgboost==1.7.6 m2cgen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import m2cgen as m2c

# =========================================================
# 1. CARREGAMENTO DO NOVO DATASET
# =========================================================
print("1. Carregando o Dataset Mestre Tabular...")
df_tabular = pd.read_csv('dataset_ia_hardware_real.csv')

# Filtramos apenas as classes 0 e 1
df_treino = df_tabular[df_tabular['label_original'] <= 1].copy()

y = df_treino['label_original'].values
colunas_features = [f'feat_{i}' for i in range(36)]
X = df_treino[colunas_features].values

# =========================================================
# 2. SEPARAÇÃO ESTRATIFICADA
# =========================================================
print("\n2. Separando Dados...")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Treino: {len(y_tr)} amostras | Teste: {len(y_te)} amostras")

# =========================================================
# 3. O CÉREBRO XGBOOST (Simples, Binário e Imbatível)
# =========================================================
print("\n3. Montando as Árvores de Decisão...")
modelo_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=15,             # 15 árvores são perfeitas para o ESP32
    learning_rate=0.1,
    max_depth=3,                 # Árvore rasa previne "decoreba"
    random_state=42,
    base_score=0.5
)

# =========================================================
# 4. TREINAMENTO
# =========================================================
print("\n Iniciando o Treinamento...")
modelo_xgb.fit(X_tr, y_tr)

previsoes_classes = modelo_xgb.predict(X_te)
nomes_classes = ['Normal', 'Desbal.']

print("\n--- RELATÓRIO DO XGBOOST ---")
print(classification_report(y_te, previsoes_classes, target_names=nomes_classes))

# =========================================================
# 5. EXPORTAÇÃO PARA O ESP32
# =========================================================
print("\n Gerando o arquivo C++ para o ESP32...")
codigo_c = m2c.export_to_c(modelo_xgb)

with open("embarcado_xgboost.h", "w") as f:
    f.write(codigo_c)

print("ARQUIVO 'embarcado_xgboost.h' GERADO COM SUCESSO!")

1. Carregando o Dataset Mestre Tabular...

2. Separando Dados...
✅ Treino: 2988 amostras | Teste: 747 amostras

3. Montando as Árvores de Decisão...

🚀 Iniciando o Treinamento...

--- RELATÓRIO DO XGBOOST ---
              precision    recall  f1-score   support

      Normal       0.93      0.99      0.96       376
     Desbal.       0.99      0.92      0.96       371

    accuracy                           0.96       747
   macro avg       0.96      0.96      0.96       747
weighted avg       0.96      0.96      0.96       747


⚡ Gerando o arquivo C++ para o ESP32...
🎉 ARQUIVO 'embarcado_xgboost.h' GERADO COM SUCESSO!


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# =========================================================
# 1. CARREGAMENTO DO DATASET
# =========================================================
print("1. Carregando o Dataset para o Especialista Cloud...")
df_tabular = pd.read_csv('dataset_ia_hardware_real.csv')

# Filtramos as classes 0 e 1 (Se futuramente adicionar outros defeitos, ajuste aqui)
df_treino = df_tabular[df_tabular['label_original'] <= 1].copy()

y = df_treino['label_original'].values
colunas_features = [f'feat_{i}' for i in range(36)]
X = df_treino[colunas_features].values

# =========================================================
# 2. SEPARAÇÃO ESTRATIFICADA
# =========================================================
print("\n2. Separando Dados...")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f" Treino: {len(y_tr)} amostras | Teste: {len(y_te)} amostras")

# =========================================================
# 3. O CÉREBRO XGBOOST DO SERVIDOR (O Especialista Parrudo)
# =========================================================
print("\n3. Montando as Árvores de Decisão de Alta Complexidade...")
modelo_xgb_servidor = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=500,          # Salto de 15 para 500 árvores
    learning_rate=0.05,        # Aprendizado mais lento e refinado
    max_depth=10,              # Árvores profundas para correlações complexas das 36 features
    subsample=0.8,             # Usa 80% dos dados por árvore (previne decoreba/overfitting)
    colsample_bytree=0.8,      # Usa 80% das features por árvore (força a IA a olhar padrões ocultos)
    random_state=42,
    base_score=0.5
)

# =========================================================
# 4. TREINAMENTO
# =========================================================
print("\n Iniciando o Treinamento no Servidor...")
modelo_xgb_servidor.fit(X_tr, y_tr)

previsoes_classes = modelo_xgb_servidor.predict(X_te)
nomes_classes = ['Normal', 'Desbal.']

print("\n--- RELATÓRIO DO XGBOOST SERVIDOR ---")
print(classification_report(y_te, previsoes_classes, target_names=nomes_classes))

# Opcional: Mostrar a Matriz de Confusão para análise
print("\nMatriz de Confusão:")
print(confusion_matrix(y_te, previsoes_classes))

# =========================================================
# 5. EXPORTAÇÃO PARA A API (Backend)
# =========================================================
print("\n Salvando o modelo compilado para produção...")

# O formato JSON é o padrão nativo moderno do XGBoost
nome_arquivo = "modelo_xgboost_especialista.json"
modelo_xgb_servidor.save_model(nome_arquivo)

print(f"ARQUIVO '{nome_arquivo}' GERADO COM SUCESSO!")

1. Carregando o Dataset para o Especialista Cloud...

2. Separando Dados...
✅ Treino: 2988 amostras | Teste: 747 amostras

3. Montando as Árvores de Decisão de Alta Complexidade...

🚀 Iniciando o Treinamento no Servidor...

--- RELATÓRIO DO XGBOOST SERVIDOR ---
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       376
     Desbal.       1.00      1.00      1.00       371

    accuracy                           1.00       747
   macro avg       1.00      1.00      1.00       747
weighted avg       1.00      1.00      1.00       747


Matriz de Confusão:
[[376   0]
 [  0 371]]

⚡ Salvando o modelo compilado para produção...
🎉 ARQUIVO 'modelo_xgboost_especialista.json' GERADO COM SUCESSO!
